# DeiT-Tiny CIFAR-100 — seed 2: projection and rollback comparison

Run this notebook independently from the other seed notebook. Attach CIFAR-100,
enable Internet and T4x2, and enable Kaggle Secret `github_token`.
Optional resume input: the output archive from this same seed/config.
Seed1 checkpoints cannot initialize seed2's experiment.

## Phase 1: Vanilla plateau
Train Vanilla from scratch for this seed; strict validation accuracy best resets
the stall. After150 epochs without exceeding the historical best, confirm plateau.
No minimum epoch and no requirement to finish the cosine schedule. Fork all four
arms from exactly that best model, AdamW and scheduler, RNG and loader state.
The150 Vanilla confirmation epochs are retained as an observed baseline.

## Phase 2: four arms, 150 training epochs each

| Arm | Projection | Rollback |
| --- | --- | --- |
| `vanilla_rollback` | None; genuine Vanilla training | Yes |
| `o_projection_only` | O-only, fixed `blocks.11.mlp` | No |
| `o_projection_only_rollback` | Same O-only as above | Yes |
| `e_driven_o_raw` | Raw E-WHERE followed by projection into O | Yes |

Vanilla with rollback is a new150-epoch continuation from the fork; the ordinary
Vanilla confirmation trajectory cannot substitute for it. This tests the
controller effect separately, and O-only with/without rollback isolates that effect.

Shared rollback rule: accuracy increases OR equal accuracy/lower loss updates
anchor; otherwise increment stall once. At10 stalled epochs restore model,
optimizer and scheduler to the latest anchor, preserving current RNG/data stream.
No new intervention on rollback. Report best uses strict accuracy only and keeps
loss from the same best epoch. Scientific escape compares post-fork best to this
seed's historical Vanilla best, never to the other seed or another arm.

Recipe: batch64, AdamW LR5e-4/WD.05, warmup5 + fixed cosine300 to LR5e-6;
bias/norm/CLS/position have no weight decay. CP: rank4, projection32,
scales(.0125,.025,.05), stats256, WHERE3x32, gate32, CG200. No random-control,
normalized-WHERE, persistent-growth or Opt-E arm runs in these notebooks.
The validation evaluation split is3000 samples;2000 reserved samples are unused
for trigger/selection. Official test is unused.

Phase1 uses GPU0; Phase2 queues four isolated arms across two GPUs, one arm per GPU.
The next queued arm fills a free GPU; no Opt-E barrier is relevant here.
Each epoch saves full state and prints method/GPU, train/val, LR, report best,
anchor, stall counters and rollback. On resume completed arms are skipped and
partial arms continue without repeating their projection. Output is archived in
`finally`; abrupt kernel termination may leave only the latest completed checkpoints.


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/deit-all-arms-repo")
OUTPUT = Path("/kaggle/working/projection_log_analysis")
URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
token = UserSecretsClient().get_secret("github_token").strip()
if not token:
    raise RuntimeError("Enable the github_token Kaggle Secret")
askpass = Path("/kaggle/working/.projection_log_git_askpass.py")
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
try:
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", URL, str(REPO)], env=env, check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], env=env, check=True)
        subprocess.run(["git", "-C", str(REPO), "merge", "--ff-only", "origin/main"], env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN_RUNTIME", None)
    del token
GROMO = Path("/kaggle/working/deit-gromo")
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
if not GROMO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/growingnet/gromo.git", str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1", "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach", GROMO_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[vision,test]", "-e", str(GROMO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
ENV = os.environ.copy()
ENV["PYTHONPATH"] = os.pathsep.join((str(REPO), str(GROMO / "src")))
ENV["REQUIRE_DEIT_INTEGRATION"] = "1"
subprocess.run([sys.executable, "-m", "pytest", "-q",
    "tests/test_deit_function_preserving.py", "tests/test_deit_sites.py",
    "tests/test_deit_projection.py", "tests/test_deit_no_aux_persistence.py",
    "tests/test_deit_protocol.py", "tests/test_deit_vanilla_reference.py", "tests/test_deit_e_rollback.py", "tests/test_deit_ablation.py", "tests/test_deit_ablation_integration.py", "tests/test_deit_logging.py", "tests/test_deit_parallel.py", "tests/test_deit_resume.py", "tests/test_kaggle_checkpoint_discovery.py"], cwd=REPO, env=ENV, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable a CUDA GPU; this notebook trains DeiT")
print("Repository commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from dataclasses import asdict
from experiments.deit_protocol import DeitRecipe, protocol, canonical_model_config
from adapters.deit_ablation import res18_cp_config
from experiments.shared_protocol import sha256_file

SEED = 2
ARMS = ('vanilla_rollback', 'o_projection_only', 'o_projection_only_rollback', 'e_driven_o_raw')
SCHEDULE_EPOCHS = 300
STALL_PATIENCE = 150
POST_FORK_EPOCHS = 150
ALGORITHM_PATIENCE = 10
MAX_EPOCH = 800
BATCH_SIZE = 64
GPU_DEVICES = 'auto'  # T4x2: two workers; one GPU: one worker.
LEARNING_RATE = 5e-4  # Explicit per-arm LR; no implicit ImageNet batch scaling.
RANK = 4
PROJECTION_SAMPLES = 32
SCALES = '.0125,.025,.05'
OUTPUT = Path(f'/kaggle/working/deit_tiny_seed{SEED}_rollback_comparison_v5')
OUTPUT.mkdir(parents=True, exist_ok=True)
RECIPE = DeitRecipe(seed=SEED, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    schedule_epochs=SCHEDULE_EPOCHS, stall_start_epoch=0,
    stall_patience=STALL_PATIENCE, reference_epochs=POST_FORK_EPOCHS, max_epoch=MAX_EPOCH)
RECIPE.validate()
EXPECTED = protocol(RECIPE, canonical_model_config())
CP = res18_cp_config()
from dataclasses import replace
CP = replace(CP, rank=RANK, projection_samples=PROJECTION_SAMPLES,
             scales=tuple(map(float, SCALES.split(','))))
roots = set()
for directory, _, files in os.walk('/kaggle/input', followlinks=True):
    if Path(directory).name == 'cifar-100-python' and {'train', 'meta'}.issubset(files):
        roots.add(Path(directory).parent.resolve())
if len(roots) != 1:
    raise FileNotFoundError(f'Attach exactly one CIFAR-100 dataset: {roots}')
DATA_ROOT = next(iter(roots))
print('Recipe:', asdict(RECIPE))
print('Projection:', asdict(CP))


In [ ]:
# Import full outputs (original .pt, .tar.gz/.zip, or mounted notebook output).
import shutil
from experiments.deit_resume import prepare_resume
RESUME_PLAN = prepare_resume(['/kaggle/input'], OUTPUT, RECIPE, CP,
    horizon=POST_FORK_EPOCHS, patience=ALGORITHM_PATIENCE, inner_steps=5, arms=ARMS)
PHASE1 = OUTPUT / 'vanilla_stall'
FORK = PHASE1 / 'plateau_checkpoint.pt'
VANILLA_REFERENCE = PHASE1 / 'vanilla_reference.pt'


In [ ]:
command = [sys.executable, '-m', 'experiments.run_deit_all_arms', '--data-root', str(DATA_ROOT),
    '--output', str(OUTPUT), '--seed', str(SEED), '--schedule-epochs', str(SCHEDULE_EPOCHS),
    '--max-epoch', str(MAX_EPOCH), '--batch-size', str(BATCH_SIZE), '--learning-rate', str(LEARNING_RATE),
    '--stall-patience', str(STALL_PATIENCE), '--post-fork-epochs', str(POST_FORK_EPOCHS),
    '--algorithm-patience', str(ALGORITHM_PATIENCE), '--rank', str(RANK),
    '--projection-samples', str(PROJECTION_SAMPLES), '--scales', SCALES, '--opt-inner-steps', '5',
    '--gpu-devices', GPU_DEVICES, '--arms', ','.join(ARMS)]
if FORK.exists():
    command += ['--plateau-checkpoint', str(FORK), '--vanilla-reference', str(VANILLA_REFERENCE)]
try:
    subprocess.run(command, cwd=REPO, env=ENV, check=True)
finally:
    archive = shutil.make_archive(str(OUTPUT), 'gztar', root_dir=OUTPUT)
    print('Resumable output archive:', archive)


In [ ]:
# Compare post-fork strict bests; never replace an arm by another arm's best.
from experiments.shared_protocol import atomic_json_save
comparison = {}
for method in ARMS:
    path = OUTPUT / 'arms' / method / 'result.json'
    if path.exists():
        result = json.loads(path.read_text())
        comparison[method] = {key: result[key] for key in (
            'fork_epoch', 'historical_best_accuracy', 'report_best_accuracy',
            'report_best_loss', 'report_best_epoch', 'delta_vs_historical_best',
            'scientific_escape', 'intervention_count')}
        comparison[method]['rollback_count'] = len(result['rollback_events'])
if 'vanilla_rollback' in comparison:
    control = comparison['vanilla_rollback']['report_best_accuracy']
    for row in comparison.values():
        row['delta_vs_vanilla_rollback'] = row['report_best_accuracy'] - control
atomic_json_save(comparison, OUTPUT / 'rollback_comparison.json')
print(json.dumps(comparison, indent=2))
status_path = OUTPUT / 'arm_status.json'
if status_path.exists():
    print('Status:', status_path.read_text())
